# Sprint & Meeting Intelligence Summerizer

In [1]:
%pip install -q openai anthropic pydantic python-dotenv pandas tiktoken


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

PROVIDER = os.getenv("PROVIDER", "openai")

## Meeting

In [3]:
ATTENDEES = ["Anita", "Rahul", "Priya", "Karthik", "Meera"]

TRANSCRIPT = """[00:00] Anita: Morning all. Sprint 14 review and planning. Let's keep it to 30 minutes.
[00:20] Meera: Quick context. The client wants the payment fixes live before their campaign.
[01:05] Anita: Release plan first. Let's release on Friday as usual.
[01:30] Rahul: Payment timeout bug is still open. I'll fix the payment timeout bug this week.
[02:10] Karthik: Staging server phir se down tha since yesterday. I'm blocked on regression testing until DevOps fixes it.
[02:45] Anita: I'll ping DevOps about staging right after this call.
[03:20] Meera: On payments, we agreed the retry limit will be 3 attempts, not 5. That's final from the client side.
[04:00] Priya: Frontend for OTP login is ready, but we still haven't received the SMS vendor API keys. OTP can't go live without them.
[04:35] Meera: I'll chase the SMS vendor for the keys by tomorrow.
[05:10] Rahul: Also, the database migration for the new invoices table is bigger than we thought. Maybe two days.
[05:50] Anita: Then Rahul, you focus on the migration. Priya, can you take the payment timeout bug instead?
[06:05] Priya: Sure, I'll have it done by Thursday.
[06:30] Rahul: Fine. I'll finish the migration script by Wednesday.
[07:15] Karthik: Maybe we should look at switching from Selenium to Playwright at some point. Tests are flaky.
[07:40] Anita: Good thought. Let's discuss that in the retro, not now.
[08:10] Meera: Dark mode story. I think we drop it from this sprint. Nobody is asking for it.
[08:25] Anita: Agreed, dark mode is out for Sprint 14.
[09:00] Priya: Someone needs to update the API docs for the new invoice endpoints.
[09:15] Anita: Yes, noted. We'll find someone.
[09:40] Rahul: Given the migration timeline, a Friday release looks risky.
[10:05] Anita: Okay, decision: we move the release to Monday. Meera, can you confirm Monday with the client today?
[10:20] Meera: Will do. By the way, I already sent the release email draft yesterday, so that's covered.
[10:45] Karthik: Once staging is back, I'll run the full regression suite before Monday.
[11:10] Anita: Great. Thanks everyone."""

print(len(TRANSCRIPT.splitlines()), "lines,", len(TRANSCRIPT), "characters")

24 lines, 2079 characters


In [4]:
GOLD = {
    "actionItems": [
        ("Priya", ["timeout"]),
        ("Rahul", ["migration"]),
        ("Anita", ["devops", "staging"]),
        ("Meera", ["sms", "vendor", "keys"]),
        ("Karthik", ["regression"]),
        (None, ["doc"])
    ],
    "decisions": [["retry"], ["dark mode"], ["monday release"]],
    "blockers": [["staging"], ["sms", "otp", "keys"]],
}

# Schema

In [5]:
from pydantic import BaseModel, Field, ValidationError
from typing import Optional

class ActionItem(BaseModel):
    task: str = Field(..., description="What must be done, starting with a verb, e.g., 'Fix the payment timeout bug'")
    owner: Optional[str] = Field(None, description="First name from the attendee list, or null if nobody took it")
    due: Optional[str] = Field(None, description="Deadline in the speaker's words, e.g., 'by Thursday', 'by next week', or null if no deadline was mentioned")
    said_at: Optional[str] = Field(None, description="Timestamp in the transcript when the action item was finally agreed, e.g., '04:35', or null if not mentioned")

class Decision(BaseModel):
    decision: str = Field(..., description="What was decided, e.g., 'Retry limit will be 3 attempts'")
    said_at: Optional[str] = Field(None, description="Timestamp in the transcript when the decision was made, e.g., '03:20', or null if not mentioned")

class Blocker(BaseModel):
    blocker: str = Field(..., description="What is blocking progress, e.g., 'Staging server down'")
    affects: Optional[str] = Field(None, description="What work is being blocked, e.g., 'Regression testing', or null if not mentioned")
    waiting_on: Optional[str] = Field(None, description="Who or what must act to unblock the work, e.g., 'DevOps', or null if not mentioned")

class MeetingSummary(BaseModel):
    executive_summary: list[str] = Field(min_length=3, max_length=3, description="A concise summary of the meeting, exactly 3 sentences")
    action_items: list[ActionItem] = Field(..., description="List of action items agreed upon in the meeting")
    decisions: list[Decision] = Field(..., description="List of decisions made in the meeting")
    blockers: list[Blocker] = Field(..., description="List of blockers mentioned in the meeting")
    open_questions: list[str] = Field(..., description="List of open questions that were raised but not answered in the meeting")

In [6]:
try:
    summary = MeetingSummary(
        executive_summary=[
            "The team discussed the Sprint 14 review and planning, focusing on the payment fixes and release schedule.",
            "Several action items were assigned, including fixing the payment timeout bug, completing the database migration script, and addressing the staging server issue.",
            ],
        action_items=[
            ActionItem(task="Fix the payment timeout bug", owner="Priya", due="by Thursday", said_at="06:05"),
            ActionItem(task="Finish the database migration script", owner="Rahul", due="by Wednesday", said_at="06:30"),
            ActionItem(task="Ping DevOps about staging server issue", owner="Anita", due=None, said_at="02:45"),
            ActionItem(task="Chase SMS vendor for API keys", owner="Meera", due="by tomorrow", said_at="04:35"),
            ActionItem(task="Run full regression suite once staging is back", owner="Karthik", due=None, said_at="10:45"),
            ActionItem(task="Update API docs for new invoice endpoints", owner=None, due=None, said_at=None),
        ],
        decisions=[
            Decision(decision="Retry limit will be 3 attempts", said_at="03:20"),
            Decision(decision="Dark mode story dropped from Sprint 14", said_at="08:25"),
            Decision(decision="Release moved to Monday", said_at="10:05"),
        ],
        blockers=[
            Blocker(blocker="Staging server down", affects="Regression testing", waiting_on="DevOps"),
            Blocker(blocker="SMS vendor API keys not received", affects="OTP login feature", waiting_on=None),
        ],
        open_questions=[]
    )
except ValidationError as e:
    print(e)

1 validation error for MeetingSummary
executive_summary
  List should have at least 3 items after validation, not 2 [type=too_short, input_value=['The team discussed the ... staging server issue.'], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/too_short


## Helpers

In [7]:
import re, json

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
CLAUDE_MODEL = os.getenv("CLAUDE_MODEL", "anthropic/claude-haiku-4-5-20251001")

def call_llm(system: str, user: str, temperature: float = 0.0) -> str:
    if PROVIDER == "openai":
        from openai import OpenAI

        resp = OpenAI().chat.completions.create(
        model=OPENAI_MODEL,
        response_format={ "type": "json_object" },
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        temperature=temperature,
        )
        print("openAI response: ", resp)
        return resp.choices[0].message.content

    if PROVIDER == "anthropic":
        import anthropic

        resp = anthropic.Anthropic().completions.create(
        model=CLAUDE_MODEL,
        extra_body={"temperature": temperature},
        max_token=1024,
        system=system,
        messages=[{"content": user, "role": "user"}],
        )
        print("Anthropic response:", resp)
        return resp.content[0].text

    raise ValueError(f"Unknown provider: {PROVIDER}")


def extract_json(text: str) -> str:
  """ Strip JSON fences and any chatter around the object """
  start, end = text.find("{"), text.rfind("}")
  return text[start:end+1] if start != -1 and end != -1 else text


def ask_json(system: str, user: str, model_cls, temperature: float = 0.0, max_retries: int = 2):
    """" Retry loop, now for any pydantic model """
    error = None

    for attempt in range(max_retries):
        prompt = user if error is None else (
            f"{user}\n\nThe previous attempt failed with the following error:\n{error}\nPlease try again and ensure the output is valid JSON that conforms to the expected schema."
        )

        raw = call_llm(system, prompt, temperature)
        try:
            return model_cls.model_validate_json(extract_json(raw))
        except ValidationError as e:
            error = str(e)
            print(f"Validation error on attempt {attempt+1}: {e}")
        except Exception as e:
            error = str(e)
            print(f"Error on attempt {attempt+1}: {e}")
    raise RuntimeError(f"Failed to get valid JSON after {max_retries} attempts. Last error: {error}")

Naive Prompt

In [8]:
import json

# if "ATTENDEES" not in globals():
#     ATTENDEES = ["Anita", "Rahul", "Priya", "Karthik", "Meera"]

# if "TRANSCRIPT" not in globals():
#     TRANSCRIPT = ""

USER_MSG = (
    f"Attendees: {', '.join(ATTENDEES)}\n\n"
    f"Transcript:\n{TRANSCRIPT}\n\n"
)

SYSTEM_V1 = f"""
Summarize this meeting. Return ONLY a JSON object matching this schema:
{json.dumps(MeetingSummary.model_json_schema(), indent=2)}
"""

v1 = ask_json(SYSTEM_V1, USER_MSG, MeetingSummary)

print("Executive Summary:", v1.model_dump_json(indent=2))

openAI response:  ChatCompletion(id='chatcmpl-EUER7hkp1mPQMCdrx3M12mKdECT3J', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n  "executive_summary": [\n    "The team reviewed Sprint 14 priorities, focusing on payment fixes, invoice migration, and release timing in light of client needs and current blockers.",\n    "They reassigned the payment timeout bug to Priya, had Rahul focus on the invoices migration, decided to move the release to Monday, and dropped dark mode from the sprint.",\n    "Key blockers include the staging server outage and missing SMS vendor API keys, with follow-ups assigned to Anita and Meera respectively."\n  ],\n  "action_items": [\n    {\n      "task": "Ping DevOps to fix the staging server so regression testing can resume",\n      "owner": "Anita",\n      "due": "right after this call",\n      "said_at": "02:45"\n    },\n    {\n      "task": "Chase the SMS vendor for the OTP API keys",\n      "owner": "Meer

In [9]:
def trap_report(s: MeetingSummary) -> dict:
    """Six traps hidden in the transcript. True = the model avoided the trap."""
    acts = s.action_items
    timeout_owners = {a.owner for a in acts if "timeout" in a.task.lower()}
    doc_items = [a for a in acts if "doc" in a.task.lower()]
    decisions = " ".join(d.decision.lower() for d in s.decisions)
    blockers = " ".join(b.blocker.lower() for b in s.blockers)
    everything = decisions + " ".join(a.task.lower() for a in acts)
    return {
        "T1 reassigned bug -> Priya":        timeout_owners == {"Priya"},
        "T2 release is Monday, not Friday":  "monday" in decisions and not re.search(r"release on friday|friday release", decisions),
        "T3 'maybe' is not a decision":      "playwright" not in everything,
        "T4 API docs left unassigned":       bool(doc_items) and all(a.owner is None for a in doc_items),
        "T5 Hinglish blocker caught":        "staging" in blockers,
        "T6 done work is not an action":     not any("email" in a.task.lower() for a in acts),
    }

def show_traps(s):
    for name, ok in trap_report(s).items():
        print(("PASS " if ok else "FAIL ") + name)

show_traps(v1)

PASS T1 reassigned bug -> Priya
PASS T2 release is Monday, not Friday
PASS T3 'maybe' is not a decision
PASS T4 API docs left unassigned
PASS T5 Hinglish blocker caught
PASS T6 done work is not an action


One rule per trap

In [10]:
RULES = """Rules:
1. An action item is something a person AGREED to do. Use the verb they used.
2. If a task is reassigned later in the meeting, use ONLY the final owner.
3. A decision is something the group agreed. If it changes later, keep ONLY the final version.
4. "Maybe we should", "at some point" and "let's discuss later" are open questions, not decisions or actions.
5. If nobody took a task ("someone needs to"), set owner to null. Never guess an owner.
6. Work that is already done is not an action item.
7. Owners must be first names from the attendee list.
8. The executive summary is exactly 3 short sentences: purpose, key outcomes, biggest risk.
9. Speakers may mix English with Hindi or Kannada. Read the meaning, not just the English words."""

SYSTEM_V2 = f"""You are a meeting analyst for a software team.
{RULES}
Return ONLY a JSON object matching this schema:
{json.dumps(MeetingSummary.model_json_schema(), indent=2)}"""

v2 = ask_json(SYSTEM_V2, USER_MSG, MeetingSummary)
show_traps(v2)

openAI response:  ChatCompletion(id='chatcmpl-EUERH6xVs9Kp8deL5JD3LOBZtk3vE', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n  "executive_summary": [\n    "The team reviewed Sprint 14 work and adjusted the release plan around payment fixes and infrastructure constraints.",\n    "Key outcomes were reassigning the payment timeout bug, prioritizing the invoice migration, moving the release to Monday, and dropping dark mode from the sprint.",\n    "The biggest risks are the staging server outage, pending SMS vendor API keys, and the tight timeline for the database migration before Monday\'s release."\n  ],\n  "action_items": [\n    {\n      "task": "Ping DevOps about staging right after this call",\n      "owner": "Anita",\n      "due": "right after this call",\n      "said_at": "02:45"\n    },\n    {\n      "task": "Chase the SMS vendor for the keys by tomorrow",\n      "owner": "Meera",\n      "due": "by tomorrow",\n      "said_at"

Chain of thoughts

In [11]:
class MeetingSummaryCoT(BaseModel):
    working_notes: str = Field(description=(
        "Think step by step BEFORE filling the summary: "
        "(a) list every candidate action, decision and blocker with its timestamp; "
        "(b) for each, check whether it is changed or reassigned LATER; "
        "(c) cross out tentative ideas and work already done."))
    summary: MeetingSummary

SYSTEM_V3 = f"""You are a meeting analyst for a software team.
{RULES}
Write your working notes first, then the final summary.
Return ONLY a JSON object matching this schema:
{json.dumps(MeetingSummaryCoT.model_json_schema(), indent=2)}"""

v3_full = ask_json(SYSTEM_V3, USER_MSG, MeetingSummaryCoT)
print(v3_full.working_notes[:1200])
v3 = v3_full.summary
show_traps(v3)

openAI response:  ChatCompletion(id='chatcmpl-EUERMB5d0gdeA9ibmYDbDU7LuXiq6', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n  "working_notes": "Attendees: Anita, Rahul, Priya, Karthik, Meera.\\n\\nStep (a) List every candidate action, decision, blocker with timestamp:\\n[01:05] Anita: \\"Let\'s release on Friday as usual.\\" -> candidate decision: release on Friday.\\n[01:30] Rahul: \\"I\'ll fix the payment timeout bug this week.\\" -> candidate action: Rahul fix payment timeout bug this week.\\n[02:10] Karthik: \\"Staging server phir se down tha since yesterday. I\'m blocked on regression testing until DevOps fixes it.\\" -> blocker: staging server down; affects regression testing; waiting_on: DevOps.\\n[02:45] Anita: \\"I\'ll ping DevOps about staging right after this call.\\" -> action: Anita ping DevOps about staging right after this call.\\n[03:20] Meera: \\"we agreed the retry limit will be 3 attempts, not 5. That\'s final

Few shots

In [12]:
# A tiny example from a DIFFERENT team (marketing), so the model copies the FORMAT, not the content.
EXAMPLE_IN = """[00:00] Leela: Newsletter planning.
[00:40] Arjun: I'll write the product section.
[01:10] Leela: Actually Sana, can you write the product section? Arjun, take the images.
[01:20] Sana: Sure, by Tuesday.
[02:00] Arjun: Maybe we should try a video newsletter someday.
[02:30] Leela: Decision: we send on the 15th, not the 10th."""

EXAMPLE_OUT = {
  "executive_summary": ["The team planned the next newsletter.",
                        "Sana now owns the product section and Arjun the images; send date is the 15th.",
                        "No blockers were raised."],
  "action_items": [
    {"task": "Write the product section", "owner": "Sana", "due": "Tuesday", "said_at": "01:20"},
    {"task": "Prepare the images", "owner": "Arjun", "due": None, "said_at": "01:10"}],
  "decisions": [{"decision": "Send the newsletter on the 15th", "said_at": "02:30"}],
  "blockers": [],
  "open_questions": ["Try a video newsletter"]}

SYSTEM_V4 = f"""You are a meeting analyst for a software team.
{RULES}

Example input:
{EXAMPLE_IN}

Example output:
{json.dumps(EXAMPLE_OUT, indent=2)}

Now do the same for the real meeting. Do not copy anything from the example.
Return ONLY a JSON object matching this schema:
{json.dumps(MeetingSummary.model_json_schema(), indent=2)}"""

v4 = ask_json(SYSTEM_V4, USER_MSG, MeetingSummary)
show_traps(v4)
leaked = [w for w in ("newsletter", "sana", "arjun", "leela") if w in v4.model_dump_json().lower()]
print("Few-shot leakage:", leaked or "none")

openAI response:  ChatCompletion(id='chatcmpl-EUEReXbz3ROUzQMQq91KUVzv4Tw0w', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n  "executive_summary": [\n    "The team reviewed Sprint 14 work and adjusted the release and scope based on current risks and client needs.",\n    "Key outcomes include moving the release to Monday, dropping dark mode from the sprint, and reassigning the payment timeout bug to Priya while Rahul focuses on the invoice migration.",\n    "The biggest risks are the staging server outage, pending SMS vendor API keys, and the tight timeline for the database migration before Monday\'s release."\n  ],\n  "action_items": [\n    {\n      "task": "Fix the payment timeout bug",\n      "owner": "Priya",\n      "due": "by Thursday",\n      "said_at": "06:05"\n    },\n    {\n      "task": "Ping DevOps about staging",\n      "owner": "Anita",\n      "due": "right after this call",\n      "said_at": "02:45"\n    },\n    {\n

Measure, don't guess

In [13]:
import pandas as pd

def _hit(text, kws):
    return any(k in text.lower() for k in kws)

def score(s: MeetingSummary) -> dict:
    found = 0
    for owner, kws in GOLD["actionItems"]:
        if any(_hit(a.task, kws) and a.owner == owner for a in s.action_items):
            found += 1
    d_found = sum(any(_hit(d.decision, k) for d in s.decisions) for k in GOLD["decisions"])
    b_found = sum(any(_hit(b.blocker, k) for b in s.blockers) for k in GOLD["blockers"])
    return {
        "actions (right owner)": f"{found}/{len(GOLD['actionItems'])}",
        "decisions": f"{d_found}/{len(GOLD['decisions'])}",
        "blockers": f"{b_found}/{len(GOLD['blockers'])}",
        "extra actions": max(0, len(s.action_items) - len(GOLD["actionItems"])),
        "traps passed": f"{sum(trap_report(s).values())}/6",
    }

ladder = pd.DataFrame({"v1 naive": score(v1), "v2 rules": score(v2),
                       "v3 + CoT": score(v3), "v4 + few-shot": score(v4)}).T
display(ladder)

,actions (right owner),decisions,blockers,extra actions,traps passed
v1 naive,6/6,2/3,2/2,1,6/6
v2 rules,5/6,2/3,2/2,1,5/6
v3 + CoT,6/6,2/3,2/2,1,6/6
v4 + few-shot,6/6,2/3,2/2,2,6/6


Tokens and COntext window

In [14]:
try:
    import tiktoken
    _enc = tiktoken.get_encoding("o200k_base")
    def count_tokens(text: str) -> int:
        return len(_enc.encode(text))
except Exception:                      # no tiktoken or no internet: rough rule
    def count_tokens(text: str) -> int:
        return len(text) // 4          # ~4 characters per English token

print("Transcript tokens:", count_tokens(TRANSCRIPT))
print("A 60-minute meeting is roughly 8,000-12,000 words, i.e. 11,000-16,000 tokens.")

Transcript tokens: 553
A 60-minute meeting is roughly 8,000-12,000 words, i.e. 11,000-16,000 tokens.


Chunk + map-reduce

In [15]:
# We use a TINY budget so our short transcript splits into chunks.
# In real life the budget would be thousands of tokens.
CHUNK_TOKENS = 170
OVERLAP_LINES = 1

def chunk_transcript(text: str, budget: int = CHUNK_TOKENS, overlap: int = OVERLAP_LINES) -> list[str]:
    """Split on speaker turns (never mid-sentence), repeat `overlap` lines between chunks."""
    lines, chunks, current = text.splitlines(), [], []
    for line in lines:
        if current and count_tokens("\n".join(current + [line])) > budget:
            chunks.append("\n".join(current))
            current = current[-overlap:] if overlap else []
        current.append(line)
    if current:
        chunks.append("\n".join(current))
    return chunks

CHUNKS = chunk_transcript(TRANSCRIPT)

for i, c in enumerate(CHUNKS, 1):
    first, last = c.splitlines()[0][:7], c.splitlines()[-1][:7]
    print(f"chunk {i}: {count_tokens(c):>4} tokens  {first} to {last}")
    print(c)

chunk 1:  170 tokens  [00:00] to [03:20]
[00:00] Anita: Morning all. Sprint 14 review and planning. Let's keep it to 30 minutes.
[00:20] Meera: Quick context. The client wants the payment fixes live before their campaign.
[01:05] Anita: Release plan first. Let's release on Friday as usual.
[01:30] Rahul: Payment timeout bug is still open. I'll fix the payment timeout bug this week.
[02:10] Karthik: Staging server phir se down tha since yesterday. I'm blocked on regression testing until DevOps fixes it.
[02:45] Anita: I'll ping DevOps about staging right after this call.
[03:20] Meera: On payments, we agreed the retry limit will be 3 attempts, not 5. That's final from the client side.
chunk 2:  158 tokens  [03:20] to [06:05]
[03:20] Meera: On payments, we agreed the retry limit will be 3 attempts, not 5. That's final from the client side.
[04:00] Priya: Frontend for OTP login is ready, but we still haven't received the SMS vendor API keys. OTP can't go live without them.
[04:35] Meera: 

In [16]:
class ChunkExtract(BaseModel):
    action_items: list[ActionItem]
    decisions: list[Decision]
    blockers: list[Blocker]
    open_questions: list[str] = Field(default_factory=list)

MAP_SYSTEM = f"""MODE: MAP. You see ONE part of a longer meeting.
Extract every candidate action item, decision, blocker and open question WITH its timestamp.
Do not resolve anything: later parts of the meeting may change these.
Return ONLY a JSON object matching this schema:
{json.dumps(ChunkExtract.model_json_schema(), indent=2)}"""

REDUCE_SYSTEM = f"""MODE: REDUCE. You receive extracts from consecutive parts of ONE meeting, in time order.
Merge them into one final summary:
- remove duplicates (chunks overlap, so the same line can appear twice)
- when a later timestamp reassigns a task or changes a decision, keep ONLY the later one
{RULES}
Return ONLY a JSON object matching this schema:
{json.dumps(MeetingSummary.model_json_schema(), indent=2)}"""

extracts = []
for i, chunk in enumerate(CHUNKS, 1):                       # MAP: one call per chunk
    ex = ask_json(MAP_SYSTEM, f"Attendees: {', '.join(ATTENDEES)}\n\nPart {i}:\n---\n{chunk}\n---", ChunkExtract)
    extracts.append(ex.model_dump())
    print(f"chunk {i}: {len(ex.action_items)} actions, {len(ex.decisions)} decisions")

final = ask_json(REDUCE_SYSTEM,                              # REDUCE: one call over all extracts
                 "Extracts in time order:\n" + json.dumps(extracts, indent=2),
                 MeetingSummary, temperature=0.0, max_retries=4)
show_traps(final)
display(pd.DataFrame({"single call (v2)": score(v2), "map-reduce": score(final)}).T)

openAI response:  ChatCompletion(id='chatcmpl-EUERksJQ9n46ljYHm2jqDsYGSm7OD', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n  "action_items": [\n    {\n      "task": "Fix the payment timeout bug",\n      "owner": "Rahul",\n      "due": "this week",\n      "said_at": "01:30"\n    },\n    {\n      "task": "Ping DevOps about the staging server issue",\n      "owner": "Anita",\n      "due": "right after this call",\n      "said_at": "02:45"\n    }\n  ],\n  "decisions": [\n    {\n      "decision": "Release will be on Friday as usual",\n      "said_at": "01:05"\n    },\n    {\n      "decision": "Payment retry limit will be 3 attempts, not 5",\n      "said_at": "03:20"\n    }\n  ],\n  "blockers": [\n    {\n      "blocker": "Staging server has been down since yesterday",\n      "affects": "Regression testing",\n      "waiting_on": "DevOps"\n    }\n  ],\n  "open_questions": []\n}', refusal=None, role='assistant', annotations=[], audio=No

,actions (right owner),decisions,blockers,extra actions,traps passed
single call (v2),5/6,2/3,2/2,1,5/6
map-reduce,6/6,2/3,2/2,4,5/6


Fact-check owners and post to slack

In [17]:
def check_owners(s: MeetingSummary, attendees: list[str]) -> list[str]:
    """Pydantic checked the shape. This checks a FACT: is every owner a real attendee?"""
    return [f"'{a.owner}' is not an attendee (task: {a.task})"
            for a in s.action_items if a.owner is not None and a.owner not in attendees]

print(check_owners(final, ATTENDEES) or "All owners are real attendees")

All owners are real attendees


In [18]:
def to_markdown(s: MeetingSummary, title: str = "Sprint 14 planning") -> str:
    out = [f"*{title}*", "", " ".join(s.executive_summary), "", "*Action items*"]
    for i, a in enumerate(s.action_items, 1):
        owner = a.owner or "UNASSIGNED"
        due = f" (by {a.due})" if a.due else ""
        out.append(f"{i}. {owner}: {a.task}{due}")
    out += ["", "*Decisions*"] + [f"- {d.decision}" for d in s.decisions]
    out += ["", "*Blockers*"] + [f"- {b.blocker}" + (f" (waiting on {b.waiting_on})" if b.waiting_on else "")
                                 for b in s.blockers]
    if s.open_questions:
        out += ["", "*Parked for later*"] + [f"- {q}" for q in s.open_questions]
    return "\n".join(out)
print(final.model_dump_json(indent=2))
print(to_markdown(final))

{
  "executive_summary": [
    "The meeting focused on release planning, ownership of critical bugs and migrations, and handling environment and vendor-related blockers.",
    "Key outcomes included moving the release from Friday to Monday, assigning Rahul to the database migration and Priya to the payment timeout bug, and dropping dark mode from Sprint 14 while defining several concrete follow-up tasks.",
    "The biggest risk is that external dependencies (staging server stability and pending SMS vendor OTP API keys) may delay regression testing and OTP login go-live."
  ],
  "action_items": [
    {
      "task": "Ping DevOps about the staging server issue",
      "owner": "Anita",
      "due": "right after this call",
      "said_at": "02:45"
    },
    {
      "task": "Chase the SMS vendor for the OTP API keys",
      "owner": "Meera",
      "due": "by tomorrow",
      "said_at": "04:35"
    },
    {
      "task": "Focus on the database migration for the new invoices table",
      